# Conversational Memory with Summarization (Groq)

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3

## Overview
Standard chatbots forget previous messages once the context window fills up. This notebook solves that by periodically compressing the conversation into a summary, which is then used as memory. The summarization is performed by Groq’s Llama, ensuring that long conversations stay coherent without blowing the token budget.

## What You Will Build
- A stateful chatbot that maintains a conversation summary.
- Automatic summarization after every 3 exchanges.
- Ability to save and load the summary for later sessions.
- Interactive loop that displays the current summary on request.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Conversational memory via summarisation.**

### Install & Imports

In [1]:
!pip install -q groq

from getpass import getpass
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.8 MB/s eta 0:00:00


### API Key & Client

In [2]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### Summarization Function

In [3]:
def summarize_conversation(history_text):
    prompt = f"""You are a summarization assistant. Condense the following conversation into a single short paragraph that captures the main topics, questions asked, and answers given. Keep it factual and concise.

Conversation:
{history_text}

Summary:
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=256
    )
    return response.choices[0].message.content.strip()

### Chatbot with Memory

In [4]:
class SummarizingChatbot:
    def __init__(self, summarization_interval=3):
        self.summary = ""
        self.history = []
        self.turn_count = 0
        self.interval = summarization_interval

    def _merge_memory(self):
        if self.summary:
            return f"Previous conversation summary: {self.summary}\n\nNew messages:\n" + "\n".join(self.history)
        else:
            return "\n".join(self.history)

    def respond(self, user_message):
        # Add user message to history
        self.history.append(f"User: {user_message}")
        self.turn_count += 1

        # Build context from memory (summary + recent messages)
        context = self._merge_memory()

        # Generate response
        prompt = f"""You are a helpful assistant. Use the conversation memory to answer consistently.
{context}
Assistant:"""
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=512
        )
        assistant_reply = response.choices[0].message.content.strip()
        self.history.append(f"Assistant: {assistant_reply}")

        # If we've reached the interval, summarise the entire history and clear it
        if self.turn_count % self.interval == 0:
            full_history = "\n".join(self.history)
            new_summary = summarize_conversation(full_history)
            if self.summary:
                self.summary = f"{self.summary} Then later: {new_summary}"
            else:
                self.summary = new_summary
            # Optionally keep only the last two exchanges after summarisation
            if len(self.history) > 4:
                self.history = self.history[-4:]   # keep last 4 lines for immediate context
        return assistant_reply

    def get_summary(self):
        return self.summary

### Interactive Chat Loop

In [5]:
brain = SummarizingChatbot(summarization_interval=3)
print("Chatbot ready. Type 'exit' to quit. Type 'summary' to see current memory.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "exit":
        break
    if user_input.lower() == "summary":
        print("\n--- Current Memory Summary ---")
        print(brain.get_summary() if brain.get_summary() else "No summary yet.")
        print("-------------------------------\n")
        continue
    if not user_input:
        continue
    reply = brain.respond(user_input)
    print(f"Bot: {reply}\n")

Chatbot ready. Type 'exit' to quit. Type 'summary' to see current memory.

You: summary

--- Current Memory Summary ---
No summary yet.
-------------------------------

You: My name is Ibrahim and I love AI.
Bot: Nice to meet you, Ibrahim. It's great to hear that you're interested in AI. What aspects of artificial intelligence fascinate you the most?

You: What is your name? I work as an AI engineer.
Bot: Nice to share that with you, Ibrahim. My name is Ada, and I'm an assistant designed to provide helpful information. That's really cool that you work as an AI engineer - what kind of projects have you been working on, and how do you think AI will evolve in the future?

You: What is the weather in UK now?
Bot: Nice question, Ibrahim. As a digital assistant, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in the UK. You can check online weather websites such as the BBC Weather or AccuWeather, which prov

### Example Demonstration

In [6]:
print("Running a short example conversation:\n")
example_bot = SummarizingChatbot(summarization_interval=2)
print("Bot: Hello! I can remember long conversations.")
messages = [
    "Hi, my name is Alice.",
    "I work as a data scientist.",
    "What is machine learning?",
    "Can you explain reinforcement learning?"
]
for msg in messages:
    print(f"User: {msg}")
    resp = example_bot.respond(msg)
    print(f"Bot: {resp}\n")
print("Final summary:", example_bot.get_summary())

Running a short example conversation:

Bot: Hello! I can remember long conversations.
User: Hi, my name is Alice.
Bot: Hello Alice, it's nice to meet you. Is there something I can help you with or would you like to chat?

User: I work as a data scientist.
Bot: That's fascinating, Alice. Data science is a field that's constantly evolving, with new techniques and technologies emerging all the time. What area of data science do you specialize in, and what kind of projects have you been working on lately?

User: What is machine learning?
Bot: Machine learning is a subset of data science that involves training algorithms to learn patterns and make predictions or decisions based on data. It's a key area of focus in data science, and it has many applications in areas such as image and speech recognition, natural language processing, and predictive modeling. As a data scientist, you may work with machine learning techniques to develop models that can classify data, make recommendations, or for

### Final Summary

In [7]:
print("Conversational Memory with Summarization - COMPLETED")
print("Author: Ibrahim")
print(" Summarises conversation every N turns.")
print(" Keeps memory without growing context window.")
print(" Ideal for long‑running chatbots.")

Conversational Memory with Summarization - COMPLETED
Author: Ibrahim
 Summarises conversation every N turns.
 Keeps memory without growing context window.
 Ideal for long‑running chatbots.
